In [1]:
from dotenv import load_dotenv,find_dotenv
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_google_genai import GoogleGenerativeAI,ChatGoogleGenerativeAI
from langchain.chains import create_sql_query_chain
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough,RunnableLambda,RunnableParallel
import re

In [2]:
load_dotenv(find_dotenv("../.env"))

True

In [3]:
db=SQLDatabase.from_uri(database_uri="sqlite:///db/chinook.db/chinook.db")

In [4]:
db.dialect

'sqlite'

In [5]:
db.get_usable_table_names()

['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

In [6]:
# llm=ChatGoogleGenerativeAI(model="gemini-1.5-flash-001",temperature=0.3)
llm=ChatOpenAI(model="gpt-3.5-turbo",temperature=0.3)

In [7]:
def extractText(string):
    return string.split(":")[-1].strip()

In [20]:
chain=create_sql_query_chain(llm=llm,db=db) | RunnableLambda(func=extractText)

In [23]:
query={"question":"How many Employees are there? Only provide the query. Provide the response without a key-value pair"}

response=chain.invoke(input=query)

In [24]:
response

'SELECT COUNT(EmployeeId) AS TotalEmployees FROM Employee;'

In [25]:
db.run(command=response)

'[(8,)]'

In [26]:
chain.get_prompts()[0].pretty_print()

You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most 5 results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result

### Using QuerySQLDataBaseTool

In [27]:
executeQuery=QuerySQLDataBaseTool(db=db)

C:\Users\MSI\AppData\Local\Temp\ipykernel_24452\1163124164.py:1: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-community package and should be used instead. To use it run `pip install -U :class:`~langchain-community` and import as `from :class:`~langchain_community.tools import QuerySQLDatabaseTool``.
  executeQuery=QuerySQLDataBaseTool(db=db)


In [29]:
writeQuery=create_sql_query_chain(llm=llm,db=db)
chain=writeQuery|executeQuery

In [30]:
chain.invoke(input=query)

'[(8,)]'

### Using Question Answering

In [32]:
answerPrompt=PromptTemplate.from_template(
    template="""
        Given the following user question, corresponding SQL query and SQL Result,
        Interpret the SQL Result from the user question:

        Question: {question}
        SQL Query: {query}
        SQL Result: {result}
        Answer: 
    """
)

answer=answerPrompt | llm | StrOutputParser() |RunnableLambda(func=lambda resp: resp.strip())

In [33]:
chain = (RunnablePassthrough.assign(query=writeQuery).assign(result=itemgetter("query") |RunnableLambda(func=extractText)| executeQuery) | answer)

In [34]:
chain.invoke(input=query)

'There are 8 Employees.'